# 🚀 End-to-End ML Pipeline: Coursera Course Recommender & Classifier
### Real-Time Training → Model Export (.pkl) → Gradio App

**Dataset:** `courses_en.csv` — 5,411 Coursera courses across 11 categories  
**Pipeline:** EDA → Preprocessing → Feature Engineering → Model Training (live progress) → Export → Gradio UI  

---
### What this notebook builds:
| Stage | Description |
|---|---|
| 📊 EDA | Dataset exploration + visualizations |
| ⚙️ Preprocessing | Text cleaning, TF-IDF, label encoding |
| 🤖 Training | Multi-model comparison with real-time Gradio progress |
| 💾 Export | Best model → `.pkl` + feature pipeline |
| 🌐 Gradio App | Live Category Classifier + Course Recommender |

## 📦 Cell 1 — Install Dependencies

In [ ]:

import subprocess, sys

packages = [
    "gradio>=4.0",
    "scikit-learn>=1.3",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "plotly",
    "nltk",
    "wordcloud",
    "joblib",
    "tqdm",
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

print("✅ All packages installed!")

## 📥 Cell 2 — Imports & Config

In [ ]:

import pandas as pd
import numpy as np
import re, os, warnings, time, json
from pathlib import Path


import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from wordcloud import WordCloud


import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)


from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.neighbors import NearestNeighbors
import joblib


import gradio as gr

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)
SEED = 42
np.random.seed(SEED)


DATA_PATH    = "courses_en.csv"    
MODEL_DIR    = Path("models")
MODEL_DIR.mkdir(exist_ok=True)
BEST_MODEL_PATH    = MODEL_DIR / "best_classifier.pkl"
VECTORIZER_PATH    = MODEL_DIR / "tfidf_vectorizer.pkl"
LABEL_ENC_PATH     = MODEL_DIR / "label_encoder.pkl"
RECOMMENDER_PATH   = MODEL_DIR / "recommender_index.pkl"
METADATA_PATH      = MODEL_DIR / "metadata.json"

print("✅ Imports complete — Gradio", gr.__version__)

## 📊 Cell 3 — Load & Explore Data (EDA)

In [ ]:
df = pd.read_csv(DATA_PATH)

print("=" * 55)
print(f"  Dataset Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Categories    : {df['category'].nunique()}")
print(f"  Languages     : {df['language'].nunique()}")
print(f"  Null values   : {df.isnull().sum().sum()}")
print("=" * 55)
display(df.head(3))
display(df.dtypes.rename('dtype').to_frame())


fig, axes = plt.subplots(1, 2, figsize=(14, 4))

missing = df.isnull().mean() * 100
axes[0].bar(missing.index, missing.values, color=['#e74c3c' if v > 0 else '#2ecc71' for v in missing.values])
axes[0].set_title('Missing Values (%)', fontsize=13, fontweight='bold')
axes[0].set_xticklabels(missing.index, rotation=30, ha='right')
axes[0].set_ylabel('%')

cat_counts = df['category'].value_counts()
axes[1].barh(cat_counts.index, cat_counts.values, color=sns.color_palette('husl', len(cat_counts)))
axes[1].set_title('Courses per Category', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Count')
for i, v in enumerate(cat_counts.values):
    axes[1].text(v + 5, i, str(v), va='center', fontsize=9)

plt.tight_layout()
plt.savefig(MODEL_DIR / 'eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ EDA overview saved")

## 🔤 Cell 4 — Text Analysis & Word Clouds

In [ ]:

df['content_len'] = df['content'].fillna('').apply(len)
df['skills_count'] = df['skills'].fillna('').apply(lambda x: len(str(x).split(',')))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df['content_len'], bins=50, color='#3498db', edgecolor='white', linewidth=0.5)
axes[0].set_title('Course Content Length Distribution', fontweight='bold')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['content_len'].median(), color='red', linestyle='--', label=f'Median: {df["content_len"].median():.0f}')
axes[0].legend()

axes[1].hist(df['skills_count'], bins=30, color='#9b59b6', edgecolor='white', linewidth=0.5)
axes[1].set_title('Number of Skills per Course', fontweight='bold')
axes[1].set_xlabel('Skills Count')
axes[1].set_ylabel('Frequency')
plt.tight_layout()
plt.show()


top_cats = df['category'].value_counts().head(3).index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
stop = set(stopwords.words('english'))

for ax, cat in zip(axes, top_cats):
    text = ' '.join(df[df['category'] == cat]['content'].fillna('').tolist())
    wc = WordCloud(
        width=500, height=300, background_color='white',
        stopwords=stop, colormap='viridis', max_words=80
    ).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(cat, fontsize=12, fontweight='bold')

plt.suptitle('Word Clouds by Top Categories', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("✅ Text analysis complete")

## ⚙️ Cell 5 — Text Preprocessing & Feature Engineering

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))

def clean_text(text: str) -> str:
    """Lowercase → remove URLs/specials → lemmatize → strip stopwords."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)       
    text = re.sub(r'[^a-z\s]', ' ', text)               
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t not in stop_words and len(t) > 2
    ]
    return ' '.join(tokens)


print("🔄 Cleaning text features...")
t0 = time.time()


df['combined_text'] = (
    df['name'].fillna('') + ' ' +
    df['skills'].fillna('') + ' ' +
    df['what_you_learn'].fillna('') + ' ' +
    df['content'].fillna('')
)
df['clean_text'] = df['combined_text'].apply(clean_text)


le = LabelEncoder()
df['label'] = le.fit_transform(df['category'])

print(f"   ✅ Text cleaned in {time.time()-t0:.1f}s")
print(f"   Classes: {list(le.classes_)}")


print("\n🔄 Fitting TF-IDF vectorizer...")
vectorizer = TfidfVectorizer(
    max_features=25_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
)
X = vectorizer.fit_transform(df['clean_text'])
y = df['label'].values

print(f"   ✅ TF-IDF matrix: {X.shape[0]:,} × {X.shape[1]:,}")


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
print(f"   Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

## 🤖 Cell 6 — Real-Time Multi-Model Training (Gradio Live Progress)
> **Run this cell — a Gradio app will open showing live training results as each model finishes.**

In [ ]:

CANDIDATE_MODELS = {
    "Logistic Regression": LogisticRegression(
        C=5, max_iter=1000, solver='lbfgs', multi_class='auto', n_jobs=-1, random_state=SEED
    ),
    "Linear SVM": LinearSVC(
        C=1.0, max_iter=2000, random_state=SEED
    ),
    "Multinomial NaiveBayes": MultinomialNB(alpha=0.1),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=25, n_jobs=-1, random_state=SEED
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.1, max_depth=5,
        subsample=0.8, random_state=SEED
    ),
}


results_log   = []   
trained_models = {}  





def run_training():
    results_log.clear()
    trained_models.clear()
    total = len(CANDIDATE_MODELS)

    for i, (name, clf) in enumerate(CANDIDATE_MODELS.items(), 1):
        status_msg = f"⏳ Training **{name}** ({i}/{total}) ..."
        yield status_msg, build_results_table(), build_bar_chart()

        t0 = time.time()
        clf.fit(X_train, y_train)
        elapsed = time.time() - t0

        y_pred = clf.predict(X_test)
        acc    = accuracy_score(y_test, y_pred)
        f1     = f1_score(y_test, y_pred, average='weighted')

        trained_models[name] = clf
        results_log.append({
            "Model": name,
            "Accuracy": round(acc * 100, 2),
            "F1-Score (weighted)": round(f1 * 100, 2),
            "Train Time (s)": round(elapsed, 1),
        })

        status_msg = f"✅ **{name}** done — Acc: {acc*100:.2f}%  F1: {f1*100:.2f}%"
        yield status_msg, build_results_table(), build_bar_chart()

    best = max(results_log, key=lambda x: x['F1-Score (weighted)'])
    final_msg = (
        f"🏆 Training complete!  "
        f"Best model → **{best['Model']}**  "
        f"(F1: {best['F1-Score (weighted)']}%)"
    )
    yield final_msg, build_results_table(), build_bar_chart()


def build_results_table():
    if not results_log:
        return pd.DataFrame(columns=["Model", "Accuracy", "F1-Score (weighted)", "Train Time (s)"])
    df_res = pd.DataFrame(results_log).sort_values('F1-Score (weighted)', ascending=False).reset_index(drop=True)
    df_res.index = df_res.index + 1
    return df_res


def build_bar_chart():
    if not results_log:
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.text(0.5, 0.5, 'Waiting for results...', ha='center', va='center', fontsize=14)
        ax.axis('off')
        return fig

    df_chart = pd.DataFrame(results_log).sort_values('F1-Score (weighted)')
    colors = ['#2ecc71' if v == df_chart['F1-Score (weighted)'].max() else '#3498db'
              for v in df_chart['F1-Score (weighted)']]

    fig, ax = plt.subplots(figsize=(8, max(3, len(df_chart) * 0.8)))
    bars = ax.barh(df_chart['Model'], df_chart['F1-Score (weighted)'], color=colors, edgecolor='white')
    ax.set_xlabel('F1-Score (weighted) %', fontsize=11)
    ax.set_title('Model Comparison — F1-Score', fontsize=13, fontweight='bold')
    ax.set_xlim(0, 105)
    for bar, val in zip(bars, df_chart['F1-Score (weighted)']):
        ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=10)
    ax.grid(axis='x', alpha=0.3)
    fig.tight_layout()
    return fig



with gr.Blocks(title="🤖 Real-Time Model Training", theme=gr.themes.Soft()) as training_app:
    gr.Markdown("# 🤖 Real-Time Multi-Model Training Dashboard")
    gr.Markdown(
        f"Training **{len(CANDIDATE_MODELS)} models** on {X_train.shape[0]:,} samples.  "
        "Watch results update live as each model finishes."
    )

    with gr.Row():
        btn_train = gr.Button("▶ Start Training", variant="primary", scale=2)
        status    = gr.Markdown("Click **Start Training** to begin.")

    with gr.Row():
        with gr.Column(scale=1):
            results_table = gr.Dataframe(
                label="📋 Results Leaderboard",
                headers=["Model", "Accuracy", "F1-Score (weighted)", "Train Time (s)"],
                interactive=False,
            )
        with gr.Column(scale=1):
            chart = gr.Plot(label="📊 Live F1-Score Chart")

    btn_train.click(
        fn=run_training,
        outputs=[status, results_table, chart],
    )

training_app.queue().launch(share=False)
print("\n⬆ Training dashboard is running above. Train first, then run Cell 7.")

## 💾 Cell 7 — Save Best Model & Full Pipeline as .pkl

In [ ]:

if not results_log:
    raise RuntimeError("❌ No results found — run Cell 6 first and complete training!")

best_name  = max(results_log, key=lambda x: x['F1-Score (weighted)'])["Model"]
best_model = trained_models[best_name]

print(f"🏆 Best Model Selected: {best_name}")
print(f"   Accuracy : {max(results_log, key=lambda x: x['Accuracy'])['Accuracy']}%")
print(f"   F1-Score : {max(results_log, key=lambda x: x['F1-Score (weighted)'])['F1-Score (weighted)']}%")


joblib.dump(best_model, BEST_MODEL_PATH, compress=3)
print(f"\n💾 Saved classifier  →  {BEST_MODEL_PATH}")


joblib.dump(vectorizer, VECTORIZER_PATH, compress=3)
print(f"💾 Saved vectorizer  →  {VECTORIZER_PATH}")


joblib.dump(le, LABEL_ENC_PATH, compress=3)
print(f"💾 Saved label encoder →  {LABEL_ENC_PATH}")


print("\n🔄 Building recommender index (k-NN)...")
X_full = vectorizer.transform(df['clean_text'])
knn = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='brute', n_jobs=-1)
knn.fit(X_full)
joblib.dump(knn, RECOMMENDER_PATH, compress=3)
print(f"💾 Saved recommender  →  {RECOMMENDER_PATH}")


meta = {
    "best_model_name"  : best_name,
    "classes"          : list(le.classes_),
    "n_features"       : vectorizer.max_features,
    "n_train"          : X_train.shape[0],
    "n_test"           : X_test.shape[0],
    "leaderboard"      : results_log,
    "columns"          : df.columns.tolist(),
}
with open(METADATA_PATH, 'w') as f:
    json.dump(meta, f, indent=2)
print(f"💾 Saved metadata     →  {METADATA_PATH}")


print("\n🔍 Verifying saved artifacts...")
_clf = joblib.load(BEST_MODEL_PATH)
_vec = joblib.load(VECTORIZER_PATH)
_le  = joblib.load(LABEL_ENC_PATH)
_test_vec = _vec.transform([clean_text("machine learning deep neural networks python")])
_pred = _clf.predict(_test_vec)
print(f"   Test prediction: 'machine learning deep neural networks python'")
print(f"   → Category: {_le.inverse_transform(_pred)[0]}")
print("\n✅ All artifacts verified and ready!")

print("\n" + "="*50)
print("📁 Saved files:")
for p in MODEL_DIR.iterdir():
    size_kb = p.stat().st_size / 1024
    print(f"   {p.name:<35} {size_kb:>8.1f} KB")

## 📈 Cell 8 — Evaluation: Confusion Matrix & Classification Report

In [ ]:

y_pred_final = best_model.predict(X_test)
class_names  = list(le.classes_)

print("=" * 55)
print(f" Best Model: {best_name}")
print("=" * 55)
print(classification_report(y_test, y_pred_final, target_names=class_names))


cm = confusion_matrix(y_test, y_pred_final)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))


sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names,
    ax=axes[0], linewidths=0.5
)
axes[0].set_title(f'{best_name}\nConfusion Matrix (counts)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].tick_params(axis='x', rotation=30)


sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Greens',
    xticklabels=class_names, yticklabels=class_names,
    ax=axes[1], linewidths=0.5
)
axes[1].set_title(f'{best_name}\nConfusion Matrix (normalized)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(MODEL_DIR / 'confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Confusion matrix saved")

## 🌐 Cell 9 — Final Gradio App: Classifier + Recommender + Explorer
> **This loads the saved `.pkl` files and launches the full production-ready app.**

In [ ]:

clf_loaded  = joblib.load(BEST_MODEL_PATH)
vec_loaded  = joblib.load(VECTORIZER_PATH)
le_loaded   = joblib.load(LABEL_ENC_PATH)
knn_loaded  = joblib.load(RECOMMENDER_PATH)

with open(METADATA_PATH) as f:
    meta_loaded = json.load(f)



print("✅ All .pkl artifacts loaded")



def predict_category(text: str):
    """Classify free-text into a course category with confidence scores."""
    if not text.strip():
        return {}, "⚠️ Please enter a description."

    cleaned = clean_text(text)
    vec     = vec_loaded.transform([cleaned])

    if hasattr(clf_loaded, 'predict_proba'):
        probs = clf_loaded.predict_proba(vec)[0]
        conf_dict = {
            le_loaded.classes_[i]: float(round(p * 100, 2))
            for i, p in enumerate(probs)
        }
    elif hasattr(clf_loaded, 'decision_function'):
        scores = clf_loaded.decision_function(vec)[0]
        scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
        conf_dict = {
            le_loaded.classes_[i]: float(round(s * 100, 2))
            for i, s in enumerate(scores)
        }
    else:
        pred = clf_loaded.predict(vec)[0]
        conf_dict = {le_loaded.classes_[pred]: 100.0}

    top_cat  = max(conf_dict, key=conf_dict.get)
    top_conf = conf_dict[top_cat]
    summary  = f"**Predicted Category:** {top_cat}  ({top_conf:.1f}% confidence)"
    return conf_dict, summary


def recommend_courses(query: str, n_results: int = 5):
    """Return top-N most similar courses using k-NN on TF-IDF vectors."""
    if not query.strip():
        return pd.DataFrame(), "⚠️ Please enter a query."

    cleaned  = clean_text(query)
    q_vec    = vec_loaded.transform([cleaned])
    dists, idxs = knn_loaded.kneighbors(q_vec, n_neighbors=n_results + 1)

    rows = []
    for dist, idx in zip(dists[0][1:], idxs[0][1:]):
        row = df.iloc[idx]
        rows.append({
            "Course Name"    : row['name'],
            "Category"       : row['category'],
            "Similarity %"   : round((1 - dist) * 100, 1),
            "Skills (preview)": ', '.join(str(row['skills']).split(',')[:4]),
            "URL"            : row['url'],
        })

    result_df = pd.DataFrame(rows[:n_results])
    info = f"✅ Top {len(result_df)} courses matching: **{query[:60]}**"
    return result_df, info


def explore_dataset(category: str, min_similarity: float):
    """Filter and display courses by category."""
    if category == "All":
        filtered = df.copy()
    else:
        filtered = df[df['category'] == category].copy()

    display_df = filtered[['name', 'category', 'skills', 'language', 'url']].head(50)
    display_df.columns = ['Course Name', 'Category', 'Skills', 'Language', 'URL']
    return display_df, f"📚 Showing {len(display_df)} of {len(filtered)} courses in '{category}'"



EXAMPLES_CLASSIFY = [
    "Learn Python for data analysis, pandas, matplotlib, machine learning",
    "Anatomy, physiology, patient care, clinical diagnosis, medical ethics",
    "Financial markets, portfolio management, risk, investment strategies",
    "Deep learning, convolutional networks, NLP transformers, PyTorch",
    "Photography, video editing, storytelling, visual arts, color theory",
]

EXAMPLES_RECOMMEND = [
    "I want to learn about artificial intelligence and neural networks",
    "Cybersecurity fundamentals for beginners network protocols",
    "Sustainable energy renewable climate change policy",
    "Public speaking leadership communication skills management",
]

ALL_CATS = ["All"] + sorted(df['category'].unique().tolist())



CSS = """
.gr-button-primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); }
.title-banner { text-align: center; padding: 10px; }
"""

with gr.Blocks(
    title="📚 Course AI",
    theme=gr.themes.Soft(primary_hue="purple", secondary_hue="blue"),
    css=CSS
) as app:

    gr.Markdown(
        f"""
        # 📚 Coursera AI Assistant
        **Model:** `{meta_loaded['best_model_name']}` &nbsp;|&nbsp;
        **Dataset:** {len(df):,} courses &nbsp;|&nbsp;
        **Categories:** {len(meta_loaded['classes'])} &nbsp;|&nbsp;
        **Features:** {meta_loaded['n_features']:,} TF-IDF
        """,
        elem_classes=["title-banner"]
    )

    with gr.Tabs():

        with gr.Tab("🔍 Category Classifier"):
            gr.Markdown(
                "Describe a course or paste any text — the model predicts the best category."
            )
            with gr.Row():
                with gr.Column(scale=2):
                    text_input = gr.Textbox(
                        label="📝 Course Description / Skills / Topic",
                        placeholder="e.g. 'machine learning python pandas deep learning neural networks'",
                        lines=5,
                    )
                    with gr.Row():
                        cls_btn = gr.Button("🔍 Classify", variant="primary", scale=2)
                        cls_clr = gr.Button("🗑 Clear", scale=1)
                    cls_status = gr.Markdown()

                with gr.Column(scale=2):
                    conf_label  = gr.Label(
                        label="📊 Category Confidence Scores",
                        num_top_classes=5
                    )

            gr.Examples(
                examples=[[e] for e in EXAMPLES_CLASSIFY],
                inputs=text_input,
                label="💡 Example Queries"
            )
            cls_btn.click(
                fn=predict_category,
                inputs=text_input,
                outputs=[conf_label, cls_status]
            )
            cls_clr.click(lambda: ("", {}, ""), outputs=[text_input, conf_label, cls_status])

        with gr.Tab("🎯 Course Recommender"):
            gr.Markdown(
                "Describe what you want to learn — the AI finds the most similar courses using cosine similarity on TF-IDF embeddings."
            )
            with gr.Row():
                rec_input = gr.Textbox(
                    label="🔎 What do you want to learn?",
                    placeholder="e.g. 'I want to learn data visualization and statistics'",
                    lines=3,
                    scale=3
                )
                n_slider = gr.Slider(
                    minimum=3, maximum=15, value=5, step=1,
                    label="📋 Number of Recommendations",
                    scale=1
                )
            rec_btn    = gr.Button("🎯 Recommend Courses", variant="primary")
            rec_status = gr.Markdown()
            rec_output = gr.Dataframe(
                label="📚 Recommended Courses",
                interactive=False,
                wrap=True
            )

            gr.Examples(
                examples=[[e, 5] for e in EXAMPLES_RECOMMEND],
                inputs=[rec_input, n_slider],
                label="💡 Example Queries"
            )
            rec_btn.click(
                fn=recommend_courses,
                inputs=[rec_input, n_slider],
                outputs=[rec_output, rec_status]
            )

        with gr.Tab("📊 Dataset Explorer"):
            gr.Markdown("Browse all courses by category.")
            with gr.Row():
                cat_dropdown = gr.Dropdown(
                    choices=ALL_CATS, value="All",
                    label="📂 Filter by Category", scale=2
                )
                sim_slider = gr.Slider(
                    minimum=0, maximum=100, value=0, step=5,
                    label="Min Similarity (unused — placeholder for future)", scale=1
                )
            explore_btn    = gr.Button("🔎 Load Courses", variant="primary")
            explore_status = gr.Markdown()
            explore_output = gr.Dataframe(
                label="📋 Course Browser",
                interactive=False,
                wrap=True
            )
            explore_btn.click(
                fn=explore_dataset,
                inputs=[cat_dropdown, sim_slider],
                outputs=[explore_output, explore_status]
            )

        with gr.Tab("ℹ️ Model Info"):
            leaderboard_df = pd.DataFrame(meta_loaded['leaderboard']).sort_values(
                'F1-Score (weighted)', ascending=False
            ).reset_index(drop=True)
            leaderboard_df.index += 1

            gr.Markdown(f"""
## 🏆 Training Leaderboard

| Field | Value |
|---|---|
| Best Model | `{meta_loaded['best_model_name']}` |
| Training Samples | {meta_loaded['n_train']:,} |
| Test Samples | {meta_loaded['n_test']:,} |
| TF-IDF Features | {meta_loaded['n_features']:,} |
| Categories | {len(meta_loaded['classes'])} |

**Categories:** {', '.join(meta_loaded['classes'])}
            """)

            gr.Dataframe(
                value=leaderboard_df,
                label="All Models Comparison",
                interactive=False
            )

            gr.Markdown("""
## 📁 Saved Artifacts
| File | Description |
|---|---|
| `models/best_classifier.pkl` | Trained classifier |
| `models/tfidf_vectorizer.pkl` | TF-IDF feature extractor |
| `models/label_encoder.pkl` | Category label encoder |
| `models/recommender_index.pkl` | k-NN recommender index |
| `models/metadata.json` | Training metadata & leaderboard |
            """)

app.launch(share=True, server_port=7860)
print("\n🚀 Full Gradio app is live! Use the tabs for Classify / Recommend / Explore.")

## ♻️ Cell 10 — Reload & Use .pkl Anywhere (Standalone Example)
> **This cell shows how to reuse the saved `.pkl` files in any script — no retraining needed.**

In [ ]:

import joblib, json, re
from pathlib import Path
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

MODEL_DIR = Path("models")


clf  = joblib.load(MODEL_DIR / "best_classifier.pkl")
vec  = joblib.load(MODEL_DIR / "tfidf_vectorizer.pkl")
le   = joblib.load(MODEL_DIR / "label_encoder.pkl")
knn  = joblib.load(MODEL_DIR / "recommender_index.pkl")
meta = json.loads((MODEL_DIR / "metadata.json").read_text())


_lem  = WordNetLemmatizer()
_stop = set(stopwords.words('english'))

def quick_clean(text):
    text = re.sub(r'[^a-z\s]', ' ', text.lower())
    return ' '.join(
        _lem.lemmatize(t) for t in text.split()
        if t not in _stop and len(t) > 2
    )


test_inputs = [
    "introduction to machine learning, regression, classification, python scikit-learn",
    "anatomy, physiology, nursing, medical imaging, patient care",
    "financial markets, portfolio optimization, risk management, derivatives",
    "solar energy, wind power, electric vehicles, climate sustainability",
    "watercolor painting, art history, sculpture, design principles",
]

print("=" * 65)
print(f" Standalone Inference — Model: {meta['best_model_name']}")
print("=" * 65)
for inp in test_inputs:
    v = vec.transform([quick_clean(inp)])
    pred = clf.predict(v)[0]
    cat  = le.inverse_transform([pred])[0]
    print(f"  Input : {inp[:55]}...")
    print(f"  → Category: {cat}")
    print()

print("✅ .pkl pipeline working perfectly — ready to deploy!")

---
## ✅ Pipeline Complete!

```
courses_en.csv
    ↓
[EDA + Text Cleaning + TF-IDF]
    ↓
[Real-Time Multi-Model Training via Gradio]
    ↓
[Best Model Auto-Selected]
    ↓
models/
  ├── best_classifier.pkl      ← predict category
  ├── tfidf_vectorizer.pkl     ← transform text
  ├── label_encoder.pkl        ← decode predictions
  ├── recommender_index.pkl    ← find similar courses
  └── metadata.json            ← leaderboard + config
    ↓
[Full Gradio App — Classifier + Recommender + Explorer]
```

**To redeploy the app later — just run Cell 9 only (everything loads from `.pkl`).**